In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("D:\PPMI_Project\data_raw\clinical")
OUT  = Path("D:\PPMI_Project\data_processed")
OUT.mkdir(parents=True, exist_ok=True)

In [3]:
def load_csv(name_like):
    for p in DATA.glob("*.csv"):
        if name_like.lower() in p.stem.lower():
            return pd.read_csv(p, low_memory=False)
    raise FileNotFoundError(f"No CSV matching: {name_like}")

def pick(colnames, *cands):
    for c in cands:
        if c in colnames: return c
    return None

In [4]:
demo = load_csv("Demographics")
status = load_csv("Participant_Status") 
visits = load_csv("Visit_Type")
up3   = load_csv("UPDRS_PartIII")
moca  = load_csv("MoCA")

In [5]:
up1 = load_csv("UPDRS_PartI")    
up2 = load_csv("UPDRS_PartII")

# normalize column names
for df_ in [up1, up2]:
    df_.columns = [c.strip().upper() for c in df_.columns]

# compute NP1TOT if missing
import numpy as np

if "NP1TOT" not in up1.columns:
    np1_items = [c for c in up1.columns if c.startswith("NP1") and c != "NP1TOT"]
    # make numeric
    for c in np1_items:
        up1[c] = pd.to_numeric(up1[c], errors="coerce")
    up1["NP1TOT"] = up1[np1_items].sum(axis=1, min_count=1)

if "NP2TOT" not in up2.columns:
    np2_items = [c for c in up2.columns if c.startswith("NP2") and c != "NP2TOT"]
    for c in np2_items:
        up2[c] = pd.to_numeric(up2[c], errors="coerce")
    up2["NP2TOT"] = up2[np2_items].sum(axis=1, min_count=1)

# keep only slim columns for merging
up1_s = up1[["PATNO","EVENT_ID","NP1TOT"]].copy()
up2_s = up2[["PATNO","EVENT_ID","NP2TOT"]].copy()
print("UPDRS I rows:", len(up1_s), "| UPDRS II rows:", len(up2_s))


UPDRS I rows: 29945 | UPDRS II rows: 32036


In [6]:
for name, df in {"demo":demo,"status":status,"visits":visits,"up3":up3,"moca":moca,"up1":up1,"up2":up2}.items():
    print("\n==", name, "==")
    print(list(df.columns)[:25])


== demo ==
['REC_ID', 'PATNO', 'EVENT_ID', 'PAG_NAME', 'INFODT', 'AFICBERB', 'ASHKJEW', 'BASQUE', 'BIRTHDT', 'SEX', 'CHLDBEAR', 'HOWLIVE', 'GAYLES', 'HETERO', 'BISEXUAL', 'PANSEXUAL', 'ASEXUAL', 'OTHSEXUALITY', 'HANDED', 'HISPLAT', 'RAASIAN', 'RABLACK', 'RAHAWOPI', 'RAINDALS', 'RANOS']

== status ==
['PATNO', 'COHORT', 'COHORT_DEFINITION', 'ENROLL_DATE', 'ENROLL_STATUS', 'STATUS_DATE', 'SCREENEDAM', 'ENROLL_AGE', 'INEXPAGE', 'AV133STDY', 'TAUSTDY', 'GAITSTDY', 'PISTDY', 'SV2ASTDY', 'NXTAUSTDY', 'DATELIG', 'PPMI_ONLINE_ENROLL', 'ENRLPINK1', 'ENRLPRKN', 'ENRLSRDC', 'ENRLNORM', 'ENRLOTHGV', 'ENRLHPSM', 'ENRLRBD', 'ENRLLRRK2']

== visits ==
['REC_ID', 'PATNO', 'EVENT_ID', 'PAG_NAME', 'INFODT', 'MODVISIT', 'MODVISTYPE', 'ORIG_ENTRY', 'LAST_UPDATE']

== up3 ==
['REC_ID', 'PATNO', 'EVENT_ID', 'PAG_NAME', 'INFODT', 'PDTRTMNT', 'PDSTATE', 'HRPOSTMED', 'HRDBSON', 'HRDBSOFF', 'PDMEDYN', 'DBSYN', 'ONOFFORDER', 'OFFEXAM', 'OFFNORSN', 'DBSOFFYN', 'DBSOFFTM', 'ONEXAM', 'ONNORSN', 'HIFUYN', 'DBSONYN'

In [7]:
import re
import numpy as np

# --- 1) Try to FIND the totals if they exist ---
up3_cols = [c for c in up3.columns]
moca_cols = [c for c in moca.columns]

np3tot_name = next((c for c in up3_cols if re.fullmatch(r'(NP3TOT|P3TOT|UPDRS3TOT|TOTAL)', c, flags=re.I)), None)
mcatot_name = next((c for c in moca_cols if re.fullmatch(r'(MCATOT|MOCA_TOTAL|TOTAL)', c, flags=re.I)), None)

print("Detected NP3 total column:", np3tot_name)
print("Detected MoCA total column:", mcatot_name)

# --- 2) If UPDRS III total not present, compute by summing NP3 item columns ---
if np3tot_name is None:
    # typical item columns look like NP3BRADY, NP3RIGID_x, NP3... (numeric)
    np3_item_cols = [c for c in up3_cols if c.startswith('NP3') and c != 'NP3TOT']
    # keep only numeric columns
    np3_numeric = [c for c in np3_item_cols if np.issubdtype(up3[c].dtype, np.number)]
    if len(np3_numeric) == 0:
        # attempt coercion if they were read as strings
        for c in np3_item_cols:
            up3[c] = pd.to_numeric(up3[c], errors='coerce')
        np3_numeric = [c for c in np3_item_cols if np.issubdtype(up3[c].dtype, np.number)]
    print("NP3 item columns considered (first 20):", np3_numeric[:20])
    up3['NP3TOT_COMPUTED'] = up3[np3_numeric].sum(axis=1, min_count=1)
    np3tot_name = 'NP3TOT_COMPUTED'
    print("Computed NP3TOT_COMPUTED.")

# --- 3) If MoCA total not present, compute from item columns starting with 'MCA' ---
if mcatot_name is None:
    # exclude obvious non-score fields (time, alt time)
    exclude = {'MCAALTTM'}
    mca_item_cols = [c for c in moca_cols if c.startswith('MCA') and c not in exclude]
    # keep numeric
    mca_numeric = []
    for c in mca_item_cols:
        if not np.issubdtype(moca[c].dtype, np.number):
            moca[c] = pd.to_numeric(moca[c], errors='coerce')
        if np.issubdtype(moca[c].dtype, np.number):
            mca_numeric.append(c)
    print("MoCA item columns considered (first 20):", mca_numeric[:20])
    moca['MCATOT_COMPUTED'] = moca[mca_numeric].sum(axis=1, min_count=1)
    mcatot_name = 'MCATOT_COMPUTED'
    print("Computed MCATOT_COMPUTED.")

# Save the chosen total column names for next cells
NP3TOT_COL = np3tot_name
MCATOT_COL = mcatot_name
print("Using UPDRS total column:", NP3TOT_COL)
print("Using MoCA total column:", MCATOT_COL)

Detected NP3 total column: NP3TOT
Detected MoCA total column: MCATOT
Using UPDRS total column: NP3TOT
Using MoCA total column: MCATOT


In [8]:
# Uppercase all
for df in [demo, status, visits, up3, moca]:
    df.columns = [c.strip().upper() for c in df.columns]

PATNO, EVENT, DATEC = "PATNO", "EVENT_ID", "INFODT"

# Clean types
for df in [demo, status, visits, up3, moca]:
    df[PATNO] = df[PATNO].astype(str).str.strip()
for df in [visits, up3, moca]:
    df[EVENT] = df[EVENT].astype(str).str.strip()
visits[DATEC] = pd.to_datetime(visits[DATEC], errors="coerce")

C:\Users\sahil\AppData\Local\Temp\ipykernel_1288\2731888304.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  visits[DATEC] = pd.to_datetime(visits[DATEC], errors="coerce")


In [9]:
# Demographics
keep_demo = [PATNO] + [c for c in ["AGE","SEX","EDUCYRS","HANDED","RACE"] if c in demo.columns]
demo_s = demo[keep_demo].drop_duplicates(PATNO)

# Label
labcol = "COHORT" if "COHORT" in status.columns else "ENROLL_STATUS"
status_s = status[[PATNO, labcol]].drop_duplicates([PATNO, labcol]).rename(columns={labcol:"LABEL"})

# Visits
vis_s = visits[[PATNO, EVENT, DATEC, "PAG_NAME"]].drop_duplicates([PATNO, EVENT])

# UPDRS III
up3_s = up3[[PATNO, EVENT, DATEC, NP3TOT_COL]].rename(columns={NP3TOT_COL:"NP3TOT"}).drop_duplicates([PATNO, EVENT])

# MoCA
moca_s = moca[[PATNO, EVENT, DATEC, MCATOT_COL]].rename(columns={MCATOT_COL:"MCATOT"}).drop_duplicates([PATNO, EVENT])

In [10]:
# pick earliest visit per subject as baseline
base_event = (vis_s.sort_values([PATNO, DATEC])
                   .groupby(PATNO, as_index=False)
                   .first()[[PATNO, EVENT, DATEC]])

base = (base_event
        .merge(up3_s, on=[PATNO, EVENT], how="left")
        .merge(moca_s, on=[PATNO, EVENT], how="left"))

df = (demo_s
      .merge(status_s, on=PATNO, how="left")
      .merge(base.drop(columns=[EVENT]), on=PATNO, how="left"))

df.head()

,PATNO,SEX,HANDED,LABEL,INFODT_x,INFODT_y,NP3TOT,INFODT,MCATOT
0,3000,0.0,1.0,2,NaT,NaN,NaN,NaN,NaN
1,3001,1.0,2.0,1,2023-08-01,NaN,NaN,NaN,NaN
2,3002,0.0,1.0,1,NaT,NaN,NaN,NaN,NaN
3,3003,0.0,1.0,1,2023-10-01,NaN,NaN,NaN,NaN
4,3004,1.0,1.0,2,2024-05-01,05/2024,9.0,05/2024,28.0


In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
from pandas.api.types import is_datetime64_any_dtype as is_datetime

# ------------------------------------------------------------
# 0) Paths (adjust if needed)
# ------------------------------------------------------------
ROOT = Path("D:/PPMI_Project")
OUT  = ROOT / "data_processed"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1) Load UPDRS Part I & II and compute NP1TOT / NP2TOT
# ------------------------------------------------------------
up1 = load_csv("UPDRS_PartI")        # robust name match (nM-EDL)
up2 = load_csv("UPDRS_PartII")       # robust name match (M-EDL)

# normalize column names
for df_ in [up1, up2]:
    df_.columns = [c.strip().upper() for c in df_.columns]

# compute NP1TOT if missing
if "NP1TOT" not in up1.columns:
    np1_items = [c for c in up1.columns if c.startswith("NP1") and c != "NP1TOT"]
    for c in np1_items:
        up1[c] = pd.to_numeric(up1[c], errors="coerce")
    up1["NP1TOT"] = up1[np1_items].sum(axis=1, min_count=1)

# compute NP2TOT if missing
if "NP2TOT" not in up2.columns:
    np2_items = [c for c in up2.columns if c.startswith("NP2") and c != "NP2TOT"]
    for c in np2_items:
        up2[c] = pd.to_numeric(up2[c], errors="coerce")
    up2["NP2TOT"] = up2[np2_items].sum(axis=1, min_count=1)

# slim views for merging
up1_s = up1[["PATNO","EVENT_ID","NP1TOT"]].copy()
up2_s = up2[["PATNO","EVENT_ID","NP2TOT"]].copy()

# ------------------------------------------------------------
# 2) Ensure dates & basic fields parsed in existing tables
#    (demo, status, visits, up3, moca must already exist)
# ------------------------------------------------------------
for d in [demo, visits, up3, moca]:
    if "INFODT" in d.columns and not is_datetime(d["INFODT"]):
        d["INFODT"] = pd.to_datetime(d["INFODT"], errors="coerce")

# ------------------------------------------------------------
# 3) Start build from STATUS (all participants), then VISITS (all visits)
# ------------------------------------------------------------
df_long = status[["PATNO"]].drop_duplicates().copy()
df_long = df_long.merge(
    visits[["PATNO","EVENT_ID","INFODT"]],
    on="PATNO", how="left"
)

# DEMOGRAPHICS (SEX, BIRTHDT)
dcols = [c for c in ["PATNO","SEX","BIRTHDT"] if c in demo.columns]
if dcols:
    demo_tmp = demo[dcols].drop_duplicates("PATNO").copy()
    if "BIRTHDT" in demo_tmp.columns:
        demo_tmp["BIRTHDT"] = pd.to_datetime(demo_tmp["BIRTHDT"], errors="coerce")
    df_long = df_long.merge(demo_tmp, on="PATNO", how="left")
else:
    df_long["SEX"] = np.nan
    df_long["BIRTHDT"] = pd.NaT

# AGE at visit
if "INFODT" in df_long.columns and "BIRTHDT" in df_long.columns:
    age_years = (df_long["INFODT"] - df_long["BIRTHDT"]).dt.days / 365.25
    df_long["AGE"] = np.floor(age_years)
else:
    df_long["AGE"] = np.nan

# Encode SEX
if "SEX" in df_long.columns:
    df_long["SEX"] = (df_long["SEX"].astype(str).str.upper()
                      .map({"M":1,"MALE":1,"F":0,"FEMALE":0}))

# ------------------------------------------------------------
# 4) Prepare UPDRS-III & MoCA slim tables
# ------------------------------------------------------------
up3_s  = up3[["PATNO","EVENT_ID","NP3TOT"]].copy() if "NP3TOT" in up3.columns else up3.copy()
moca_s = moca[["PATNO","EVENT_ID","MCATOT"]].copy() if "MCATOT" in moca.columns else moca.copy()

# ------------------------------------------------------------
# 5) Harmonize merge keys BEFORE all merges (fixes dtype mismatch)
# ------------------------------------------------------------
tables = [df_long, up1_s, up2_s, up3_s, moca_s]
for t in tables:
    # PATNO as nullable integer
    t["PATNO"] = pd.to_numeric(t["PATNO"], errors="coerce").astype("Int64")
    # EVENT_ID as trimmed string (codes like BL, V01, V20, etc.)
    if "EVENT_ID" in t.columns:
        t["EVENT_ID"] = t["EVENT_ID"].astype(str).str.strip()

# ------------------------------------------------------------
# 6) Merge clinical scores
# ------------------------------------------------------------
df_long = df_long.merge(up3_s, on=["PATNO","EVENT_ID"], how="left")
df_long = df_long.merge(moca_s, on=["PATNO","EVENT_ID"], how="left")
df_long = df_long.merge(up1_s, on=["PATNO","EVENT_ID"], how="left")
df_long = df_long.merge(up2_s, on=["PATNO","EVENT_ID"], how="left")

# ------------------------------------------------------------
# 7) Build Y_PD label (robust from STATUS text fields)
# ------------------------------------------------------------
s = status.copy()
s.columns = [c.upper() for c in s.columns]
combo_cols = [c for c in [
    "COHORT","COHORT_DEFINITION","ENROLL_STATUS",
    "PRIMARY CLINICAL DIAGNOSIS","PRIMARY_CLINICAL_DIAGNOSIS","DIAGNOSIS"
] if c in s.columns]
s["_TXT_"] = ""
for c in combo_cols:
    s["_TXT_"] = s["_TXT_"] + " " + s[c].astype(str)

txt = s["_TXT_"].str.upper().fillna("")
is_pd = (
    txt.str.contains("PARKINSON") |
    txt.str.contains(r"\bPD\b", regex=True) |
    txt.str.contains("PATIENT") |
    txt.str.contains("CASE")
)
is_nonpd = (
    txt.str.contains("HEALTHY") |
    txt.str.contains("CONTROL") |
    txt.str.contains(r"\bHC\b", regex=True) |
    txt.str.contains("NON[- ]?PD")
)

s_lab = s[["PATNO"]].drop_duplicates().copy()
s_lab["PATNO"] = pd.to_numeric(s_lab["PATNO"], errors="coerce").astype("Int64")
s_lab["Y_PD"] = np.where(is_pd & ~is_nonpd, 1, 0)

df_long = df_long.merge(s_lab, on="PATNO", how="left")
df_long["Y_PD"] = df_long["Y_PD"].fillna(0).astype(int).clip(0,1)

# ------------------------------------------------------------
# 8) Save and report
# ------------------------------------------------------------
outpath = OUT / "clinical_longitudinal.csv"
df_long.to_csv(outpath, index=False)

print("Saved ->", outpath)
print("Rows:", len(df_long))
print("Label counts:\n", df_long["Y_PD"].value_counts(dropna=False))
print("Columns now:",
      [c for c in ["NP1TOT","NP2TOT","NP3TOT","MCATOT","AGE","SEX"] if c in df_long.columns])

C:\Users\sahil\AppData\Local\Temp\ipykernel_1288\1689593190.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  demo_tmp["BIRTHDT"] = pd.to_datetime(demo_tmp["BIRTHDT"], errors="coerce")


Saved -> D:\PPMI_Project\data_processed\clinical_longitudinal.csv
Rows: 17414
Label counts:
 Y_PD
0    10613
1     6801
Name: count, dtype: int64
Columns now: ['NP1TOT', 'NP2TOT', 'NP3TOT', 'MCATOT', 'AGE', 'SEX']
